# CORE Ingestion — Bronze → Silver

Reads each .tab file individually (preserving its own header), filters to London, selects common columns by name, then unions all years.

In [1]:
import os
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'
os.environ['PYSPARK_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, when, avg, lit
from pyspark.sql.functions import udf, round as spark_round
from pyspark.sql.types import FloatType
import glob

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('core_ingest') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Spark ready')

Spark ready


In [2]:
def safe_int(c):
    return when(trim(col(c)) == '', None).otherwise(trim(col(c))).cast('int')

def band_midpoint(s):
    if s is None: return None
    s = s.strip()
    if s in ('', 'Missing', 'MISSING', 'R', 'NULL', 'N/A', 'No', 'Yes', 'Refused'): return None
    if 'More than' in s or 'more than' in s:
        import re; nums = re.findall(r'[\d,]+', s)
        return float(nums[0].replace(',','')) if nums else None
    if 'Less than' in s or 'less than' in s:
        import re; nums = re.findall(r'[\d,]+', s)
        return float(nums[0].replace(',','')) / 2 if nums else None
    if ' to ' in s:
        parts = s.split(' to ')
        try: return (float(parts[0].strip().replace(',','')) + float(parts[1].strip().replace(',',''))) / 2
        except: return None
    try: return float(s.replace(',',''))
    except: return None

midpoint_udf = udf(band_midpoint, FloatType())
print('Helpers ready')

Helpers ready


In [3]:
BRONZE = '../data/bronze/core_raw/tab'
SILVER_PATH = '../data/silver/core'

all_files = sorted(glob.glob(f'{BRONZE}/*.tab'))
print(f'Total files: {len(all_files)}')

def process_file(path):
    """Read one tab file, filter to London, return standardised DataFrame (or None if no London rows)."""
    df = spark.read.csv(path, header=True, inferSchema=False, sep='\t')
    cols = set(df.columns)

    # GOVREG is numeric '7' in old files, 'E12000007' in new files
    df = df.filter(
        (trim(col('GOVREG')) == '7') | (trim(col('GOVREG')) == 'E12000007')
    )
    if df.head(1) == []:
        return None

    # Helper: safe column (null if missing)
    def sc(name):
        return trim(col(name)) if name in cols else lit(None).cast('string')

    def si(name):
        return safe_int(name) if name in cols else lit(None).cast('int')

    # BED_MINUS_BEDSTANDARD: name varies by year
    if 'BED_MINUS_BEDSTANDARD' in cols:
        bed_diff = safe_int('BED_MINUS_BEDSTANDARD')
    elif 'BED_MINUS_BEDSTANDARD2' in cols:
        bed_diff = safe_int('BED_MINUS_BEDSTANDARD2')
    else:
        bed_diff = lit(None).cast('int')

    # WTSHORTFALL band: name changed between years
    if 'WTSHORTFALLHB_Bands' in cols:
        shortfall = trim(col('WTSHORTFALLHB_Bands'))
    elif 'WTSHORTFALL_Bands' in cols:
        shortfall = trim(col('WTSHORTFALL_Bands'))
    else:
        shortfall = lit(None).cast('string')

    empstat = sc('econstat_imputed_R')
    ten_len = sc('TENANCYLENGTH_Bands')
    year_col = 'YEAR' if 'YEAR' in cols else None

    return df.select(
        (safe_int(year_col) if year_col else lit(None).cast('int')).alias('year'),
        trim(col('GOVREG')).alias('region_code_raw'),
        sc('LETTYPE').alias('let_type'),
        sc('TENANCY').alias('tenancy_type'),
        si('HHMEMBT').alias('household_size'),
        si('BEDST').alias('bedroom_standard'),
        bed_diff.alias('bedrooms_vs_need'),
        sc('PREVTEN_R').alias('previous_tenure'),
        sc('REASON_R').alias('reason_for_letting'),
        sc('WEEKINC_T_Bands').alias('weekly_income_band'),
        sc('WRENT_Bands').alias('weekly_rent_band'),
        shortfall.alias('rent_shortfall_band'),
        sc('ETHNIC_Bands').alias('ethnicity'),
        ten_len.alias('tenancy_length_band'),
        empstat.alias('employment_status'),
    )

dfs = []
for path in all_files:
    name = os.path.basename(path)
    result = process_file(path)
    if result is not None:
        n = result.count()
        print(f'{name}: {n:,} London rows')
        dfs.append(result)
    else:
        print(f'{name}: 0 London rows (skipped)')

print(f'\nFiles with London data: {len(dfs)}')

Total files: 42
0708_sr_gn_eul.tab: 17,312 London rows
0708_sr_sh_eul.tab: 11,871 London rows
0809_sr_gn_eul.tab: 18,741 London rows
0809_sr_sh_eul.tab: 11,439 London rows
0910_sr_gn_eul.tab: 15,546 London rows
0910_sr_sh_eul.tab: 10,088 London rows
1011_sr_gn_eul.tab: 18,035 London rows
1011_sr_sh_eul.tab: 13,593 London rows
1112_ar_gn_eul.tab: 614 London rows
1112_ar_sh_eul.tab: 0 London rows (skipped)
1112_sr_gn_eul.tab: 31,029 London rows
1112_sr_sh_eul.tab: 16,427 London rows
1213_ar_gn_eul.tab: 3,809 London rows
1213_ar_sh_eul.tab: 49 London rows
1213_sr_gn_eul.tab: 29,019 London rows
1213_sr_sh_eul.tab: 15,878 London rows
1314_ar_gn_eul.tab: 4,960 London rows
1314_ar_sh_eul.tab: 47 London rows
1314_sr_gn_eul.tab: 24,368 London rows
1314_sr_sh_eul.tab: 14,824 London rows
1415_ar_gn_eul.tab: 6,167 London rows
1415_ar_sh_eul.tab: 47 London rows
1415_sr_gn_eul.tab: 22,753 London rows
1415_sr_sh_eul.tab: 14,016 London rows
1516_ar_gn_eul.tab: 6,532 London rows
1516_ar_sh_eul.tab: 499

In [4]:
from functools import reduce
from pyspark.sql import DataFrame

combined = reduce(DataFrame.union, dfs)

silver = (
    combined
    .withColumn('region_code', lit('E12000007'))
    .drop('region_code_raw')
    .withColumn('weekly_income_est',  midpoint_udf(col('weekly_income_band')))
    .withColumn('weekly_rent_est',    midpoint_udf(col('weekly_rent_band')))
    .withColumn('rent_shortfall_est', midpoint_udf(col('rent_shortfall_band')))
    .withColumn(
        'rent_to_income_pct',
        when(
            (col('weekly_income_est') > 0) & col('weekly_rent_est').isNotNull(),
            (col('weekly_rent_est') / col('weekly_income_est')) * 100
        ).otherwise(None)
    )
    .withColumn('overcrowded', when(col('bedrooms_vs_need') < 0, True).otherwise(False))
)

print(f'Total London rows across all years: {silver.count():,}')
display(silver.groupBy('year').count().orderBy('year').limit(30).toPandas().style.format(thousands=","))

Total London rows across all years: 501,244
+----+-----+
|year|count|
+----+-----+
|2007|22050|
|2008|29455|
|2009|27759|
|2010|29800|
|2011|45033|
|2012|49405|
|2013|44093|
|2014|43021|
|2015|43485|
|2016|33477|
|2017|27825|
|2018|25991|
|2019|28973|
|2020|22079|
|2021|23356|
|2022| 5442|
+----+-----+



In [ ]:
display(silver.agg(
    spark_round(avg('weekly_rent_est'), 2).alias('avg_weekly_rent_£'),
    spark_round(avg('weekly_income_est'), 2).alias('avg_weekly_income_£'),
    spark_round(avg('rent_to_income_pct'), 1).alias('avg_rent_to_income_%')
).toPandas().style.format(thousands=","))

print('\nYears with rent data:')
display(silver.filter(col('weekly_rent_est').isNotNull()) \
      .groupBy('year').count().orderBy('year').limit(30).toPandas().style.format(thousands=","))

In [6]:
silver.write \
    .mode('overwrite') \
    .partitionBy('year') \
    .parquet(SILVER_PATH)

verify = spark.read.parquet(SILVER_PATH)
print(f'Written: {verify.count():,} rows')
display(verify.groupBy('year').count().orderBy('year').limit(30).toPandas().style.format(thousands=","))

Written: 501,244 rows
+----+-----+
|year|count|
+----+-----+
|2007|22050|
|2008|29455|
|2009|27759|
|2010|29800|
|2011|45033|
|2012|49405|
|2013|44093|
|2014|43021|
|2015|43485|
|2016|33477|
|2017|27825|
|2018|25991|
|2019|28973|
|2020|22079|
|2021|23356|
|2022| 5442|
+----+-----+

